# 02 - Train a DQN Model Offline

This notebook shows the offline training workflow in Mouse Core:

1. Load previously collected `Datastore` streams from the Hub.
2. Build a `DataLoader` that samples fixed-length sequences from those streams.
3. Assemble a `Model` from an embedder, a backbone, and an action-value head.
4. Train the model with `DqnObjective` and save the checkpoint with `push_model_to_hub`.

The dataset comes from the collection notebook, but the same Mouse Core pieces apply to any sequential environment data stored as step dictionaries.

During training, the loop periodically runs **greedy eval** on a separate held-out `mouse-gym` environment group (not the Hub replay streams) and prints mean task score.


In [ ]:
import torch

import procedural_frozenlake  # noqa: F401 — registers Procedural-FrozenLake-v1
from mouse_core.task_eval import (
    DEFAULT_EVAL_SEED_OFFSET,
    make_procedural_frozenlake_group,
    run_task_eval,
)

from mouse_core.data import (
    DataLoader,
    Augmenter,
    load_stores_from_hub,
)
from mouse_core.objectives import DqnObjective
from mouse_core.models import Model, preferred_dtype, push_model_to_hub
from mouse_core.models.backbone import Qwen3Backbone
from mouse_core.models.embedding import NumericEmbedder
from mouse_core.models.heads import DiscreteActionValueHead


DATASET_ID = "mouse-example-dataset"   # Hugging Face dataset repo for load_stores_from_hub
MODEL_ID = "mouse-example-model-offline"       # Hugging Face model repo for push_model_to_hub
MAX_ACTIONS = 4                        # number of discrete actions predicted by the head
MAX_OBS_DISCRETE = 64                  # vocabulary size for discrete observations
GRADIENT_STEPS = 20000                 # total optimizer updates
SEQUENCE_LENGTH = 512                  # replay sequence length sampled by DataLoader
BATCH_SIZE = 4                         # sequences per optimizer step
EVAL_EVERY = 1000                  # gradient steps between held-out evals
EVAL_NUM_ENVS = 4                  # separate eval env streams (not used for replay)
EVAL_TASKS_PER_ENV = 1             # tasks per eval env each eval round
EVAL_EPISODES_PER_TASK = 20        # env task length (points per task scale)
EVAL_MAX_EPISODE_STEPS = 30


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


## Load Data

`load_stores_from_hub` downloads the dataset snapshot and reconstructs the saved `Datastore` objects. Each returned store is one ordered environment stream.


In [ ]:
stores = load_stores_from_hub(DATASET_ID, split="train", force_download=True)

## DataLoader And Augmentation

`DataLoader` samples sequence batches from one or more datastores. With `pack=True`, it can combine shorter segments and returns parallel `segment_ids` so the model can isolate attention/RoPE per pack slice and objectives can ignore transitions that cross packed segments.

`next_batch()` returns `(batch, segment_ids)` where `batch` is `list[list[dict]]` shaped `[batch][sequence]` and `segment_ids` is a parallel `list[list[int]]` labeling which pack slice each step came from.

`Augmenter` can transform fields as batches are sampled. A modality spec names a row field and an augmentation type, such as `discrete`, `linear`, or `image`. Use `keep_fields` for values that should pass through unchanged because an objective still needs them. Segment IDs are returned separately from row dicts and are not part of `keep_fields`.


In [ ]:
augmenter = Augmenter(
    modalities=[
        {
            "field": "action",
            "type": "discrete",
            "vocab_size": MAX_ACTIONS,
            "permute": True,
        },
        {
            "field": "observation",
            "type": "discrete",
            "vocab_size": MAX_OBS_DISCRETE,
            "permute": True,
        },
    ],
    keep_fields=["action", "observation", "reward", "done"],
)

loader = DataLoader(
    stores,
    sequence_length=SEQUENCE_LENGTH,
    batch_size=BATCH_SIZE,
    prefetch=4,
    num_workers=0,
    pack=True,
    augmenter=augmenter,
)

## Evaluation environments

Training reads Hub replay only. Progress is measured on a **separate** held-out
`GroupEnv` (different `reset_seed` / `map_seed` stream via
`DEFAULT_EVAL_SEED_OFFSET`) so eval maps are not the collection set.


In [ ]:
eval_env = make_procedural_frozenlake_group(
    num_envs=EVAL_NUM_ENVS,
    episodes_per_task=EVAL_EPISODES_PER_TASK,
    max_episode_steps=EVAL_MAX_EPISODE_STEPS,
    seed_offset=DEFAULT_EVAL_SEED_OFFSET,
    name_prefix="eval_frozenlake",
)


## Build The Model

A Mouse Core `Model` has three main pieces:

- `NumericEmbedder` converts fields from each step dictionary into model tokens.
- `Qwen3Backbone` processes those tokens with a transformer backbone.
- `DiscreteActionValueHead` predicts one value per discrete action.

The backbone exposes `hidden_dim`, and the embedder and head use that same value so the pieces connect cleanly.

`NumericEmbedder` modality types used here:

- `discrete` for integer IDs such as actions, observations, and done codes.
- `rff` for scalar numeric values such as rewards.
- `learnable` when you want extra learned tokens that are not tied to a row field.

`Model(...)` wraps the pieces behind a single forward call that returns predictions, objective data, and an optional cache.


In [ ]:
backbone = Qwen3Backbone(pretrained="Qwen/Qwen3-0.6B")

encoder = NumericEmbedder(
    hidden_dim=backbone.hidden_dim,
    modalities=[
        {
            "field": "action",
            "type": "discrete",
            "vocab_size": MAX_ACTIONS,
            "std": 0.02,
            "tokens": 1
        },
        {
            "field": "observation",
            "type": "discrete",
            "vocab_size": MAX_OBS_DISCRETE,
            "std": 0.02,
            "tokens": 1
        },
        {
            "field": "reward",
            "type": "rff",
            "std": 0.02,
            "in_min": 0.01,
            "in_max": 100.0,
            "tokens": 1
        },
        {
            "field": "done",
            "type": "discrete",
            "vocab_size": 5,
            "std": 0.02,
            "tokens": 1
        },
    ],
    modality_fusion="sum",
    include_type_token=False,
)

head = DiscreteActionValueHead(
    in_features=backbone.hidden_dim,
    out_features=MAX_ACTIONS,
    hidden_dim=backbone.hidden_dim,
    num_layers=1,
    scale=0.1,
)

model = Model(encoder=encoder, backbone=backbone, heads=head).train().to(device=device, dtype=preferred_dtype(device))
model.backbone.model.compile()  # optional compile step for faster repeated forwards
print(model)


## Train

The offline training loop is intentionally small because the Mouse Core abstractions do most of the work:

1. `loader.next_batch()` samples step sequences and parallel `segment_ids`.
2. `model(batch, segment_ids=segment_ids)` embeds the dictionaries, runs the backbone with same-segment attention/RoPE, and produces head predictions.
3. `objective(objective_data, predictions)` computes the DQN loss and metrics.
4. The optimizer updates model weights.
5. `model.polyak_update(...)` updates target-network weights used by the objective.

`DqnObjective` interprets the integer `done` field through separate discount factors for normal steps, episode boundaries, and task boundaries. Set those gammas to match the semantics of your environment data.


In [ ]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1.0e-5,
    weight_decay=0.0,
    betas=(0.9, 0.95),
    eps=1.0e-8
)
objective = DqnObjective(
    gamma_step=1.0,                 # discount for ordinary next-step bootstrapping
    gamma_episode_terminal=1.0,     # discount when an episode ends normally
    gamma_episode_truncated=1.0,
    gamma_task_terminal=0.0,        # discount when a task ends normally
    gamma_task_truncated=0.0,
    tau=0.0005,
)

for step in range(GRADIENT_STEPS):

    batch, segment_ids = loader.next_batch()

    predictions, objective_data, _ = model(batch, segment_ids=segment_ids)
    loss, metrics = objective(objective_data, predictions)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    model.polyak_update(action_value_tau=objective.tau)

    if step % 100 == 0:
        print(f"step={step}  loss={loss.item():.4f}  q={metrics['q_values_mean']:.3f}")

    if step % EVAL_EVERY == 0:
        model.eval()
        stats = run_task_eval(
            model,
            eval_env,
            tasks_per_env=EVAL_TASKS_PER_ENV,
            max_cache=SEQUENCE_LENGTH,
        )
        print(
            f"  eval@{step}  mean_task_score={stats['mean_task_score']:.2f}"
            f"/{EVAL_EPISODES_PER_TASK}  n_tasks={stats['n_tasks']}"
        )
        model.train()

loader.close()
eval_env.close()


## Push To The Hub

`push_model_to_hub` saves the model architecture and weights together. Later, `load_model` can reconstruct the full `Model` without repeating the embedder, backbone, and head definitions.


In [ ]:
model.eval().to("cpu")
url = push_model_to_hub(model, MODEL_ID, private=False, clear=True)
print(f"Pushed to {url}")